# ⚽ FIFA World Cup 2026 Prediction Model (Pure NumPy Pipeline)
This Jupyter notebook contains a complete machine learning pipeline and Monte Carlo tournament simulator to predict the winner of the upcoming FIFA World Cup 2026. 

This notebook uses a **custom Poisson Regression model implemented in pure NumPy**. This design avoids dependencies on C-extension libraries like `scikit-learn` that may be blocked by system application control policies, ensuring it is 100% runnable in restricted environments.

### **Methodology Outline**
1. **Data Preprocessing & Cleaning**: Load match results, shootout outcomes, and rankings, and perform team name cleaning to establish clean linkage.
2. **Feature Engineering**: Link matches to their historical FIFA rankings (dating back to 1993) and build advanced features like relative rank differences, point differences, neutrality factor, and rolling team form.
3. **Custom Model Training**: Train a custom Poisson Regressor in pure NumPy to predict goals scored by each team based on team parameters and matchup characteristics.
4. **Monte Carlo Simulation**: Run a complete simulator of the 48-team 2026 World Cup bracket (Group stage tables, best-3rd selection, Round of 32 matching solver, and knockout bracket) 10,000 times.
5. **Visualization**: Plot predictions and export top contenders.

## 1. Import Dependencies and Load Datasets

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Load datasets
results_df = pd.read_csv('results.csv')
rankings_df = pd.read_csv('fifa_mens_rank.csv')
shootouts_df = pd.read_csv('shootouts.csv')
fixtures_df = pd.read_csv('FIFA2026_schedule_Fixtures.csv')
eafc_df = pd.read_csv('eafc26_wc_team_summary.csv')

# Apply name cleaning to EAFC data and build lookup dictionary
eafc_df['Nation'] = eafc_df['Nation'].apply(clean_name)
eafc_stats = {}
for idx, row in eafc_df.iterrows():
    eafc_stats[row['Nation']] = {
        'Top11_OVR': float(row['Top11_OVR']),
        'Attack_Score': float(row['Attack_Score']),
        'Defense_Score': float(row['Defense_Score']),
        'Penalty_Score': float(row['Penalty_Score'])
    }

def get_squad_features(team, rank):
    stats = eafc_stats.get(team)
    if stats:
        return stats['Top11_OVR'], stats['Attack_Score'], stats['Defense_Score'], stats['Penalty_Score']
    else:
        # Fallback using regression formulas derived from WC team data
        top11_ovr = np.clip(85.67 - 0.1827 * rank, 55.0, 88.5)
        atk_score = np.clip(76.43 - 0.1265 * rank, 50.0, 83.0)
        def_score = np.clip(70.69 - 0.0891 * rank, 50.0, 82.0)
        pen_score = np.clip(58.0 - 0.1 * rank, 40.0, 75.0)
        return top11_ovr, atk_score, def_score, pen_score
print("Datasets loaded successfully.")

## 2. Team Name Normalization
Standardize team names across all datasets to ensure seamless linkage.

In [ ]:
TEAM_MAPPING = {
    'USA': 'United States',
    'IR Iran': 'Iran',
    'Congo DR': 'DR Congo',
    'Cabo Verde': 'Cape Verde',
    'Korea Republic': 'South Korea',
    'Côte d'Ivoire': 'Ivory Coast',
    'Türkiye': 'Turkey',
    'Trkiye': 'Turkey',
    'Trkiye': 'Turkey',
    'Czechia': 'Czech Republic',
    'Aotearoa New Zealand': 'New Zealand',
    'Curaao': 'Curacao',
    'Curaçao': 'Curacao'
}

def clean_name(name):
    if not isinstance(name, str): return name
    name = name.strip()
    if 'Cura' in name: return 'Curacao'
    if 'Côte' in name or 'Cte' in name: return 'Ivory Coast'
    if 'Türk' in name or 'Trkiye' in name or 'Trkiye' in name: return 'Turkey'
    return TEAM_MAPPING.get(name, name)

results_df['home_team'] = results_df['home_team'].apply(clean_name)
results_df['away_team'] = results_df['away_team'].apply(clean_name)
rankings_df['team'] = rankings_df['team'].apply(clean_name)
shootouts_df['home_team'] = shootouts_df['home_team'].apply(clean_name)
shootouts_df['away_team'] = shootouts_df['away_team'].apply(clean_name)
shootouts_df['winner'] = shootouts_df['winner'].apply(clean_name)

results_df['date'] = pd.to_datetime(results_df['date'])
shootouts_df['date'] = pd.to_datetime(shootouts_df['date'])
print("Name normalization complete.")

## 3. Build Historical Rankings Database & Penalty Stats

In [ ]:
shootout_stats = {}
for idx, row in shootouts_df.iterrows():
    h, a, w = row['home_team'], row['away_team'], row['winner']
    for t in [h, a]:
        if t not in shootout_stats: shootout_stats[t] = {'wins': 0, 'total': 0}
        shootout_stats[t]['total'] += 1
        if t == w: shootout_stats[t]['wins'] += 1

def get_shootout_win_rate(team):
    stats = shootout_stats.get(team)
    return stats['wins'] / stats['total'] if stats and stats['total'] > 0 else 0.5

rankings_db = {}
for idx, row in rankings_df.iterrows():
    year, sem, team, rank, pts = int(row['date']), int(row['semester']), row['team'], int(row['rank']), float(row['total.points'])
    if team not in rankings_db: rankings_db[team] = {}
    rankings_db[team][(year, sem)] = (rank, pts)

max_year = rankings_df['date'].max()
max_sem = rankings_df[rankings_df['date'] == max_year]['semester'].max()

def get_rank_and_points(team, date):
    year, sem = date.year, (1 if date.month <= 6 else 2)
    if year > max_year or (year == max_year and sem > max_sem): year, sem = max_year, max_sem
    db = rankings_db.get(team)
    if not db: return 150, 1000.0
    if (year, sem) in db: return db[(year, sem)]
    past_keys = [k for k in db.keys() if k[0] < year or (k[0] == year and k[1] <= sem)]
    if past_keys:
        best_k = max(past_keys, key=lambda x: (x[0], x[1]))
        return db[best_k]
    return 150, 1000.0

## 4. Feature Engineering and Form Calculations

In [ ]:
results_df = results_df.sort_values('date').reset_index(drop=True)
team_match_history = {}
form_span = 10
training_rows = []

historical_matches = results_df[
    (results_df['date'] >= '1993-08-01') & 
    (results_df['home_score'].notna()) & 
    (results_df['away_score'].notna()) &
    (results_df['home_score'] != 'NA') &
    (results_df['away_score'] != 'NA')
].copy()

historical_matches['home_score'] = pd.to_numeric(historical_matches['home_score'])
historical_matches['away_score'] = pd.to_numeric(historical_matches['away_score'])

for idx, row in results_df.iterrows():
    date = row['date']
    h, a = row['home_team'], row['away_team']
    h_score_raw, a_score_raw = row['home_score'], row['away_score']
    
    is_historical = True
    try:
        hs, as_ = float(h_score_raw), float(a_score_raw)
        if np.isnan(hs) or np.isnan(as_): is_historical = False
    except (ValueError, TypeError): is_historical = False
        
    def get_form(team):
        hist = team_match_history.get(team, [])
        if not hist: return 1.2, 1.2
        recent = hist[-form_span:]
        return np.mean([x[0] for x in recent]), np.mean([x[1] for x in recent])

    if is_historical and date >= pd.to_datetime('1993-08-01'):
        hs, as_ = int(hs), int(as_)
        h_rank, h_pts = get_rank_and_points(h, date)
        a_rank, a_pts = get_rank_and_points(a, date)
        h_form_att, h_form_def = get_form(h)
        a_form_att, a_form_def = get_form(a)
        h_ovr, h_atk, h_def, h_pen = get_squad_features(h, h_rank)
        a_ovr, a_atk, a_def, a_pen = get_squad_features(a, a_rank)
        neutral = 1 if str(row['neutral']).upper() == 'TRUE' else 0
        is_friendly = 1 if row['tournament'] == 'Friendly' else 0
        
        training_rows.append({
            'team': h, 'opp': a, 'is_home': 1 - neutral,
            'rank_diff': (h_rank - a_rank) / 100.0, 'point_diff': (h_pts - a_pts) / 500.0,
            'squad_atk_self': (h_atk - 75.0) / 10.0, 'squad_def_opp': (a_def - 70.0) / 10.0, 'squad_ovr_diff': (h_ovr - a_ovr) / 10.0,
            'form_attack_self': h_form_att, 'form_defense_opp': a_form_def,
            'is_friendly': is_friendly, 'goals': hs
        })
        training_rows.append({
            'team': a, 'opp': h, 'is_home': 0,
            'rank_diff': (a_rank - h_rank) / 100.0, 'point_diff': (a_pts - h_pts) / 500.0,
            'squad_atk_self': (a_atk - 75.0) / 10.0, 'squad_def_opp': (h_def - 70.0) / 10.0, 'squad_ovr_diff': (a_ovr - h_ovr) / 10.0,
            'form_attack_self': a_form_att, 'form_defense_opp': h_form_def,
            'is_friendly': is_friendly, 'goals': as_
        })
        
    if is_historical:
        hs, as_ = int(hs), int(as_)
        if h not in team_match_history: team_match_history[h] = []
        if a not in team_match_history: team_match_history[a] = []
        team_match_history[h].append((hs, as_))
        team_match_history[a].append((as_, hs))

train_df = pd.DataFrame(training_rows)
print(f"Prepared {len(train_df)} dataset entries.")

## 5. Train Poisson Regressor (Pure NumPy GLM)

In [ ]:
class PurePoissonRegression:
    def __init__(self, lr=0.001, iterations=1500):
        self.lr = lr
        self.iterations = iterations
        self.weights = None
        self.bias = None
        
    def fit(self, X, y):
        N, D = X.shape
        self.weights = np.zeros(D)
        self.bias = 0.0
        for _ in range(self.iterations):
            linear = np.dot(X, self.weights) + self.bias
            linear = np.clip(linear, -10.0, 5.0)
            lambdas = np.exp(linear)
            dw = np.dot(X.T, (lambdas - y)) / N
            db = np.mean(lambdas - y)
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
    def predict(self, X):
        linear = np.dot(X, self.weights) + self.bias
        linear = np.clip(linear, -10.0, 5.0)
        return np.exp(linear)

features_cols = ['is_home', 'rank_diff', 'point_diff', 'squad_atk_self', 'squad_def_opp', 'squad_ovr_diff', 'form_attack_self', 'form_defense_opp', 'is_friendly']
X_df = train_df[features_cols]
y_df = train_df['goals']

np.random.seed(42)
indices = np.arange(len(train_df))
np.random.shuffle(indices)
split_idx = int(len(train_df) * 0.8)
train_idx, val_idx = indices[:split_idx], indices[split_idx:]

X_train = X_df.iloc[train_idx].to_numpy(dtype=np.float64)
y_train = y_df.iloc[train_idx].to_numpy(dtype=np.float64)
X_val = X_df.iloc[val_idx].to_numpy(dtype=np.float64)
y_val = y_df.iloc[val_idx].to_numpy(dtype=np.float64)

model = PurePoissonRegression(lr=0.01, iterations=1500)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
mae = np.mean(np.abs(y_val - y_pred))
print(f"Validation Mean Absolute Error (MAE): {mae:.4f}")

# Retrain on whole dataset
X_all = X_df.to_numpy(dtype=np.float64)
y_all = y_df.to_numpy(dtype=np.float64)
model.fit(X_all, y_all)

## 6. Build the World Cup Bracket Simulator

In [ ]:
latest_date = pd.to_datetime('2026-06-16')

# Extract group fixtures
wc_matches_all = results_df[(results_df['date'].dt.year == 2026) & (results_df['tournament'] == 'FIFA World Cup')].copy()
wc_matches = wc_matches_all[wc_matches_all['home_score'].isna() | (wc_matches_all['home_score'].astype(str) == 'NA')].reset_index(drop=True)
wc_played_matches = wc_matches_all[wc_matches_all['home_score'].notna() & (wc_matches_all['home_score'].astype(str) != 'nan') & (wc_matches_all['home_score'].astype(str) != 'NA')].copy()

fixtures_group_map = {}
for idx, row in fixtures_df.iterrows():
    date_dt, teams_str, group = row['date_dt'], row['teams'], row['group']
    if 'Group' in str(group):
        parts = teams_str.split(' v ')
        if len(parts) == 2:
            t1, t2 = clean_name(parts[0].strip()), clean_name(parts[1].strip())
            fixtures_group_map[(date_dt, t1, t2)] = group
            fixtures_group_map[(date_dt, t2, t1)] = group

team_groups = {}
group_teams = {}
for idx, row in wc_matches_all.iterrows():
    date_str = row['date'].strftime('%Y-%m-%d')
    h, a = row['home_team'], row['away_team']
    
    # Robust matching logic for play-off combinations
    group = None
    for key, g in fixtures_group_map.items():
        if date_str == key[0]:
            # check if h matches key[1] or key[2]
            def match_playoff(t, f_str):
                opts = [clean_name(opt.strip()).lower() for opt in f_str.split('/')]
                return clean_name(t).lower() in opts
            t1_m_1 = match_playoff(h, key[1])
            t2_m_2 = match_playoff(a, key[2])
            t1_m_2 = match_playoff(h, key[2])
            t2_m_1 = match_playoff(a, key[1])
            if (t1_m_1 and t2_m_2) or (t1_m_2 and t2_m_1):
                group = g
                break
                
    if not group: group = 'Group A'
    team_groups[h] = group
    team_groups[a] = group
    if group not in group_teams:
        group_teams[group] = set()
    group_teams[group].add(h)
    group_teams[group].add(a)

for g in group_teams: group_teams[g] = sorted(list(group_teams[g]))
['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curacao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan'] = sorted(list(team_groups.keys()))

# Precalculate all expected goals to speed up simulation
precalc_features = []
precalc_keys = []
for team_a in ['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curacao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan']:
    rank_a, pts_a = get_rank_and_points(team_a, latest_date)
    att_a, def_a = get_latest_form(team_a)
    ovr_a, atk_a, def_a_sq, pen_a = get_squad_features(team_a, rank_a)
    for team_b in ['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curacao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan']:
        if team_a == team_b: continue
        rank_b, pts_b = get_rank_and_points(team_b, latest_date)
        att_b, def_b = get_latest_form(team_b)
        ovr_b, atk_b, def_b_sq, pen_b = get_squad_features(team_b, rank_b)
        precalc_features.append([0.0, (rank_a - rank_b)/100.0, (pts_a - pts_b)/500.0, (atk_a - 75.0)/10.0, (def_b_sq - 70.0)/10.0, (ovr_a - ovr_b)/10.0, att_a, def_b, 0.0])
        precalc_keys.append((team_a, team_b, 1))
        precalc_features.append([1.0, (rank_a - rank_b)/100.0, (pts_a - pts_b)/500.0, (atk_a - 75.0)/10.0, (def_b_sq - 70.0)/10.0, (ovr_a - ovr_b)/10.0, att_a, def_b, 0.0])
        precalc_keys.append((team_a, team_b, 0))

precalc_preds = model.predict(np.array(precalc_features))
lambda_cache = {}
for idx, key in enumerate(precalc_keys): lambda_cache[key] = max(0.1, precalc_preds[idx])

def get_cached_lambdas(team_a, team_b, neutral):
    l_a = lambda_cache.get((team_a, team_b, neutral), 1.2)
    l_b = lambda_cache.get((team_b, team_a, neutral), 1.2)
    return l_a, l_b

def simulate_match(team_a, team_b, neutral=1, is_knockout=False):
    lambda_a, lambda_b = get_cached_lambdas(team_a, team_b, neutral)
    goals_a, goals_b = np.random.poisson(lambda_a), np.random.poisson(lambda_b)
    if not is_knockout: return goals_a, goals_b
    if goals_a == goals_b:
        goals_a += np.random.poisson(lambda_a / 3.0)
        goals_b += np.random.poisson(lambda_b / 3.0)
    if goals_a == goals_b:
        rank_a, _ = get_rank_and_points(team_a, latest_date)
        rank_b, _ = get_rank_and_points(team_b, latest_date)
        p_win_a = np.clip(0.5 + 0.2*(get_shootout_win_rate(team_a) - get_shootout_win_rate(team_b)) + 0.05*((rank_b - rank_a)/(rank_b + rank_a)), 0.35, 0.65)
        if np.random.random() < p_win_a: return goals_a + 1, goals_b
        else: return goals_a, goals_b + 1
    return goals_a, goals_b

In [ ]:
def simulate_group_stage():
    standings = {g: {t: {'pts': 0, 'gd': 0, 'gs': 0, 'team': t} for t in teams} for g, teams in group_teams.items()}
    for idx, row in wc_played_matches.iterrows():
        h, a = clean_name(row['home_team']), clean_name(row['away_team'])
        if h in team_groups and a in team_groups:
            g = team_groups[h]
            gh, ga = int(float(row['home_score'])), int(float(row['away_score']))
            standings[g][h]['gs'] += gh
            standings[g][h]['gd'] += (gh - ga)
            standings[g][a]['gs'] += ga
            standings[g][a]['gd'] += (ga - gh)
            if gh > ga: standings[g][h]['pts'] += 3
            elif gh < ga: standings[g][a]['pts'] += 3
            else:
                standings[g][h]['pts'] += 1
                standings[g][a]['pts'] += 1
    for idx, row in wc_matches.iterrows():
        h, a = row['home_team'], row['away_team']
        neutral = 1 if str(row['neutral']).upper() == 'TRUE' else 0
        g = team_groups[h]
        gh, ga = simulate_match(h, a, neutral, is_knockout=False)
        standings[g][h]['gs'] += gh
        standings[g][h]['gd'] += (gh - ga)
        standings[g][a]['gs'] += ga
        standings[g][a]['gd'] += (ga - gh)
        if gh > ga: standings[g][h]['pts'] += 3
        elif gh < ga: standings[g][a]['pts'] += 3
        else:
            standings[g][h]['pts'] += 1
            standings[g][a]['pts'] += 1
    ranked_groups = {}
    for g in group_teams:
        def sort_key(t):
            rank, _ = get_rank_and_points(t['team'], latest_date)
            return (-t['pts'], -t['gd'], -t['gs'], rank)
        ranked_groups[g] = sorted(list(standings[g].values()), key=sort_key)
    return ranked_groups

def pair_third_place_teams(third_place_teams):
    slots = [
        {'id': 74, 'allowed': {'Group A', 'Group B', 'Group C', 'Group D', 'Group F'}},
        {'id': 77, 'allowed': {'Group C', 'Group D', 'Group F', 'Group G', 'Group H'}},
        {'id': 79, 'allowed': {'Group C', 'Group E', 'Group F', 'Group H', 'Group I'}},
        {'id': 80, 'allowed': {'Group E', 'Group H', 'Group I', 'Group J', 'Group K'}},
        {'id': 81, 'allowed': {'Group B', 'Group E', 'Group F', 'Group I', 'Group J'}},
        {'id': 82, 'allowed': {'Group A', 'Group E', 'Group H', 'Group I', 'Group J'}},
        {'id': 85, 'allowed': {'Group E', 'Group F', 'Group G', 'Group I', 'Group J'}},
        {'id': 87, 'allowed': {'Group D', 'Group E', 'Group I', 'Group J', 'Group L'}}
    ]
    assigned = [None] * len(slots)
    used = set()
    def backtrack(slot_idx):
        if slot_idx == len(slots): return True
        slot = slots[slot_idx]
        for team in third_place_teams:
            if team not in used:
                t_grp = team_groups[team]
                if t_grp in slot['allowed']:
                    assigned[slot_idx] = team
                    used.add(team)
                    if backtrack(slot_idx + 1): return True
                    used.remove(team)
                    assigned[slot_idx] = None
        return False
    if backtrack(0): return {slots[i]['id']: assigned[i] for i in range(len(slots))}
    return {slots[i]['id']: third_place_teams[i] for i in range(len(slots))}

def simulate_tournament():
    group_tables = simulate_group_stage()
    winners = {g: table[0]['team'] for g, table in group_tables.items()}
    runners_up = {g: table[1]['team'] for g, table in group_tables.items()}
    third_placed = []
    for g, table in group_tables.items():
        t = table[2]
        rank, _ = get_rank_and_points(t['team'], latest_date)
        third_placed.append({'team': t['team'], 'pts': t['pts'], 'gd': t['gd'], 'gs': t['gs'], 'rank': rank})
    ranked_third = sorted(third_placed, key=lambda t: (-t['pts'], -t['gd'], -t['gs'], t['rank']))
    best_8_third_teams = [t['team'] for t in ranked_third[:8]]
    third_pairings = pair_third_place_teams(best_8_third_teams)
    
    ko_winners = {}
    matches = [
        (73, runners_up['Group A'], runners_up['Group B']),
        (74, winners['Group E'], third_pairings[74]),
        (75, winners['Group F'], runners_up['Group C']),
        (76, winners['Group C'], runners_up['Group F']),
        (77, winners['Group I'], third_pairings[77]),
        (78, runners_up['Group E'], runners_up['Group I']),
        (79, winners['Group A'], third_pairings[79]),
        (80, winners['Group L'], third_pairings[80]),
        (81, winners['Group D'], third_pairings[81]),
        (82, winners['Group G'], third_pairings[82]),
        (83, runners_up['Group K'], runners_up['Group L']),
        (84, winners['Group H'], runners_up['Group J']),
        (85, winners['Group B'], third_pairings[85]),
        (86, winners['Group J'], runners_up['Group H']),
        (87, winners['Group K'], third_pairings[87]),
        (88, runners_up['Group D'], runners_up['Group G'])
    ]
    for mid, t1, t2 in matches:
        g1, g2 = simulate_match(t1, t2, neutral=1, is_knockout=True)
        ko_winners[mid] = t1 if g1 > g2 else t2
        
    r16_brackets = [(89, 74, 77), (90, 73, 75), (91, 76, 78), (92, 79, 80), (93, 83, 84), (94, 81, 82), (95, 86, 88), (96, 85, 87)]
    for mid, m1, m2 in r16_brackets:
        g1, g2 = simulate_match(ko_winners[m1], ko_winners[m2], neutral=1, is_knockout=True)
        ko_winners[mid] = ko_winners[m1] if g1 > g2 else ko_winners[m2]
        
    qf_brackets = [(97, 89, 90), (98, 93, 94), (99, 91, 92), (100, 95, 96)]
    for mid, m1, m2 in qf_brackets:
        g1, g2 = simulate_match(ko_winners[m1], ko_winners[m2], neutral=1, is_knockout=True)
        ko_winners[mid] = ko_winners[m1] if g1 > g2 else ko_winners[m2]
        
    sf_brackets = [(101, 97, 98), (102, 99, 100)]
    for mid, m1, m2 in sf_brackets:
        g1, g2 = simulate_match(ko_winners[m1], ko_winners[m2], neutral=1, is_knockout=True)
        ko_winners[mid] = ko_winners[m1] if g1 > g2 else ko_winners[m2]
        
    g1, g2 = simulate_match(ko_winners[101], ko_winners[102], neutral=1, is_knockout=True)
    winner = ko_winners[101] if g1 > g2 else ko_winners[102]
    return {'winner': winner, 'finalists': [ko_winners[101], ko_winners[102]]}
print("Tournament simulator functions defined.")

## 7. Run Monte Carlo Simulations & Plot Winner Probabilities

In [ ]:
NUM_SIMULATIONS = 10000
winner_counts = {t: 0 for t in ['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curacao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan']}
final_counts = {t: 0 for t in ['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curacao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan']}

print(f"Simulating {NUM_SIMULATIONS} tournaments...")
for i in range(NUM_SIMULATIONS):
    res = simulate_tournament()
    winner_counts[res['winner']] += 1
    for t in res['finalists']:
        final_counts[t] += 1

wc_prob_df = pd.DataFrame({
    'Team': ['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curacao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan'],
    'Group': [team_groups[t] for t in ['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curacao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan']],
    'FIFA Rank': [get_rank_and_points(t, latest_date)[0] for t in ['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curacao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan']],
    'Final Reach (%)': [final_counts[t] / NUM_SIMULATIONS * 100 for t in ['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curacao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan']],
    'Winner (%)': [winner_counts[t] / NUM_SIMULATIONS * 100 for t in ['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curacao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan']]
}).sort_values(by='Winner (%)', ascending=False).reset_index(drop=True)

plt.figure(figsize=(12, 7))
top_15 = wc_prob_df.head(15)
ax = sns.barplot(data=top_15, x='Winner (%)', y='Team', palette='viridis', hue='Team', legend=False)
plt.title('FIFA World Cup 2026 Prediction Winner Probabilities (Top 15)', fontsize=15, fontweight='bold')
plt.xlabel('Win Probability (%)')
plt.ylabel('Team')
for p in ax.patches:
    ax.text(p.get_width() + 0.1, p.get_y() + p.get_height()/2, f"{p.get_width():.2f}%", ha='left', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

print("Top 15 Contenders:")
print(wc_prob_df.head(15))